# HALI — Health & Wellbeing
## HPV Vaccine Companion for Kenya

**Community contribution by Stella Oiro**

Built on Lab 4 (tool use + agent loop) and Lab 3 (evaluator-rerun pattern).

## Live demo
**[https://huggingface.co/spaces/AcharO/hali-hpv-kenya](https://huggingface.co/spaces/AcharO/hali-hpv-kenya)**

## The problem
Cervical cancer kills **3,400 Kenyan women every year** — Kenya's leading cancer killer of women.
The HPV vaccine is 98% effective, free at government facilities, and Kenya switched to a single-dose
schedule in October 2025. Yet uptake remains at ~60% nationally and **below 1%** in North Eastern counties.

The primary barrier is not access — it is **misinformation** and **low awareness**.

## What this builds
A dual-mode AI companion called **HALI** (Health & Wellbeing in Swahili):
- **Caregiver mode** — warm nurse persona for families, speaks English/Swahili
- **CHW mode** — clinical support for Community Health Workers in the field

## Patterns used (from the labs)
- **Tool use + agent loop** (Lab 4/5): record interest, check eligibility, log unknown questions → Pushover notifications
- **Evaluator-rerun loop** (Lab 3): second model checks every response for cultural fit and accuracy

## Project structure
Logic is modularised into:
- `prompts.py` — system prompts and Kenya HPV facts
- `tools.py` — tool functions, JSON schemas, dispatcher
- `evaluator.py` — evaluator and rerun logic
- `app.py` — Gradio UI and agent loop (run this directly)
- `tests/test_tools.py` — unit tests

This notebook walks through each piece and runs a live demo.

## Context
Built in the spirit of work done by [KEPRECON](https://keprecon.org) (Kenya Paediatric Research Consortium),
which has championed HPV vaccination advocacy across Kenya's 47 counties.

In [ ]:
from dotenv import load_dotenv
import os
import sys

# Add parent directory so modules resolve correctly when running from the notebook
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

load_dotenv(override=True)

openai_key = os.getenv('OPENAI_API_KEY')
print(f"OpenAI key: {openai_key[:8]}..." if openai_key else "OpenAI key not set")

## 1. Prompts

Two system prompts live in `prompts.py`:
- **CAREGIVER_SYSTEM_PROMPT** — warm nurse persona, English/Swahili
- **CHW_SYSTEM_PROMPT** — clinical, concise, field-focused

Both are grounded in real Kenya data: coverage stats, documented myths, trusted information sources.

In [ ]:
from prompts import CAREGIVER_SYSTEM_PROMPT, CHW_SYSTEM_PROMPT, KENYA_HPV_FACTS

print(KENYA_HPV_FACTS)

## 2. Tools

Three tools give the agent real-world reach (`tools.py`):

| Tool | What it does |
|---|---|
| `record_interest` | Logs a caregiver who wants follow-up → Pushover notification |
| `record_unknown_question` | Logs questions the agent can't answer → Pushover notification |
| `check_eligibility` | Checks HPV vaccine eligibility under Kenya's programme |

In [ ]:
from tools import check_eligibility, record_interest, TOOLS

# Test eligibility check directly
print(check_eligibility(age=12, gender="female"))
print(check_eligibility(age=8, gender="female"))
print(check_eligibility(age=20, gender="mwanamke"))  # Swahili for woman

## 3. Evaluator (Lab 3 pattern)

A second model call checks every reply before it reaches the user (`evaluator.py`).
It rejects responses that are factually wrong, culturally inappropriate, or preachy.
If rejected, `rerun()` retries with the feedback injected into the system prompt.

In [ ]:
from evaluator import Evaluation, evaluate

# Inspect the Pydantic schema
print(Evaluation.model_json_schema())

## 4. The Agent Loop (Lab 4/5 pattern)

```
User message
  → LLM called with tools
      → tool_calls? run them, feed results back, loop
      → normal reply? evaluate it
          → pass? return to user
          → fail? rerun with feedback
```

The full loop lives in `app.py`. Here we wire it up for the demo.

In [ ]:
from openai import OpenAI
from evaluator import evaluate, rerun
from tools import TOOLS, handle_tool_calls

client = OpenAI()


def chat(message: str, history: list, mode: str = "caregiver") -> str:
    system_prompt = CAREGIVER_SYSTEM_PROMPT if mode == "caregiver" else CHW_SYSTEM_PROMPT
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]

    done = False
    while not done:
        response = client.chat.completions.create(model="gpt-4o-mini", messages=messages, tools=TOOLS)
        finish_reason = response.choices[0].finish_reason

        if finish_reason == "tool_calls":
            message_obj = response.choices[0].message
            results = handle_tool_calls(message_obj.tool_calls)
            messages.append(message_obj)
            messages.extend(results)
        else:
            done = True

    reply = response.choices[0].message.content

    evaluation = evaluate(reply, message, history)
    if evaluation.is_acceptable:
        print("Passed evaluation")
    else:
        print(f"Failed: {evaluation.feedback}")
        reply = rerun(reply, message, history, evaluation.feedback, system_prompt)

    return reply

## 5. Live tests

In [ ]:
# Most common myth in Kenya — infertility fear
reply = chat("Nilisikia chanjo hii inafanya wasichana kushindwa kupata watoto. Ni kweli?", [], mode="caregiver")
print(reply)

In [ ]:
# Eligibility check
reply = chat("My daughter is 13. Is she eligible for the HPV vaccine?", [], mode="caregiver")
print(reply)

In [ ]:
# CHW mode — religious objection in North Eastern Kenya
reply = chat("A mother in Wajir says the vaccine is haram and refuses. What are my talking points?", [], mode="chw")
print(reply)

## 6. Gradio UI

Two tabs — one for families, one for health workers.
To run as a standalone app: `python app.py`

In [ ]:
import gradio as gr

def chat_caregiver(message, history):
    return chat(message, history, mode="caregiver")

def chat_chw(message, history):
    return chat(message, history, mode="chw")

with gr.Blocks(title="HALI — HPV Kenya") as demo:
    gr.Markdown(
        """# HALI — Health & Wellbeing
### HPV Vaccine Companion for Kenya

Cervical cancer kills **3,400 Kenyan women every year**. The vaccine is **free, safe, and one dose is enough.**
"""
    )
    with gr.Tabs():
        with gr.Tab("For Families (Caregivers)"):
            gr.Markdown("Ask HALI anything about the HPV vaccine — in English or Swahili.")
            gr.ChatInterface(
                fn=chat_caregiver,
                type="messages",
                examples=[
                    "I heard this vaccine makes girls unable to have babies. Is this true?",
                    "My daughter is 13. Is she eligible for the vaccine?",
                    "Where can I get the vaccine in Garissa?",
                    "Our imam says we should not take it. What do you say?",
                ],
            )
        with gr.Tab("For Health Workers (CHW)"):
            gr.Markdown("Evidence-based support for CHWs in the field.")
            gr.ChatInterface(
                fn=chat_chw,
                type="messages",
                examples=[
                    "A mother in Mandera refuses — says it's haram. How do I respond?",
                    "What is the evidence behind the single-dose schedule change?",
                    "A girl aged 16 missed the school programme. Is she still eligible?",
                ],
            )

demo.launch()

## 7. Running the tests

Unit tests cover eligibility logic, tool dispatch, push notifications, and error handling.
No API calls are made — tools that fire push notifications are mocked.

In [ ]:
!pytest tests/test_tools.py -v

## Ideas for further development

- **WhatsApp integration** — most Kenyan caregivers are on WhatsApp, not web browsers
- **County-specific clinic finder** — tool that returns nearest HPV vaccination point by county
- **RAG knowledge base** — feed in KEPRECON research, MoH guidelines, KENITAG documents
- **Swahili-first mode** — for rural users with limited English
- **Reporting dashboard** — aggregate unknown questions and interest records for KEPRECON field teams

Evidence base: The Shanghai chatbot RCT (Nature Medicine, 2025) showed a **3.85x uplift** in
vaccination rates using a similar conversational AI — and **8.81x in rural areas**.
Kenya's North Eastern counties (below 1% coverage) are exactly that context.